<a href="https://colab.research.google.com/github/llelus/DSA-Project/blob/main/notebooks/02_eda_hypothesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — EDA and Hypothesis Tests

Notation (all on a 1-minute grid, per market, strictly before expiry):
`price` = Polymarket YES price, `fair` = BTC-implied probability of YES
(log-normal, trailing 60-min realised vol, time to settlement), `gap = fair − price`,
`dp` / `d_fair` = 1-minute changes. **Lag k > 0 always means BTC at t, Polymarket at t + k.**

Inference uses the "uncertain" band 0.05 ≤ price ≤ 0.95 where the price can still move,
and standard errors are clustered by expiry hour (markets expiring together share one BTC path).

In [ ]:
# Setup — identical in every notebook; works in Colab and locally
import os, sys, subprocess
BRANCH = "main"   # GitHub branch Colab runs; it must contain src/
if "google.colab" in sys.modules:
    REPO = "/content/DSA-Project"
    URL = "https://github.com/llelus/DSA-Project.git"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, URL, REPO], check=True)
    else:  # runtime reused: make sure the requested branch is checked out and current
        subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO, "pull", "-q", "--autostash", "origin", BRANCH], check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if not os.path.isdir("src"):
    raise RuntimeError(f"src/ not found in {os.getcwd()}: branch '{BRANCH}' on GitHub does not "
                       "contain the v2 code yet (push/merge it, or set BRANCH).")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from src import config, dataset, analysis
from src.results import save_metrics, savefig

panel = dataset.add_market_diffs(dataset.load_panel())
usable = panel["fair"].notna() & (panel["tau_min"] >= config.MIN_TAU_MIN)
mid = usable & panel["price"].between(config.MID_LOW, config.MID_HIGH)
print(f"Panel: {len(panel):,} market-minutes | {panel.condition_id.nunique():,} markets | "
      f"{panel.market_end.nunique():,} expiry hours")
print(f"Usable rows: {usable.sum():,} | in the {config.MID_LOW}-{config.MID_HIGH} price band: {mid.sum():,}")

## Dataset overview

In [ ]:
per_market = panel.groupby("condition_id").size()
overview = dict(
    n_market_minutes=len(panel), n_markets=int(panel.condition_id.nunique()),
    n_expiry_hours=int(panel.market_end.nunique()),
    first_expiry=str(panel.market_end.min()), last_expiry=str(panel.market_end.max()),
    minutes_per_market_median=float(per_market.median()),
    share_mid_band=float(mid.sum() / usable.sum()),
    share_fresh_price=float((panel.stale_min < 1).mean()),
    yes_rate=float(panel.groupby("condition_id").result_yes_won.first().mean()))
save_metrics("dataset", **overview)
pd.Series(overview)

## EDA 1 — One market: Polymarket price vs BTC-implied fair value

In [ ]:
cid = panel[mid].groupby("condition_id").size().idxmax()
ex = panel[panel.condition_id == cid]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ex.ts, ex.price, lw=1.8, label="Polymarket YES price")
ax.plot(ex.ts, ex.fair, lw=1.4, ls="--", label="BTC-implied fair probability")
ax.set_ylim(0, 1); ax.set_ylabel("Probability")
ax2 = ax.twinx()
ax2.plot(ex.ts, ex.btc, color="gray", alpha=.5, lw=1)
ax2.axhline(ex.strike.iloc[0], color="gray", ls=":", lw=1)
ax2.set_ylabel("BTC/USDT (grey), strike (dotted)")
ax.set_title(f"Strike {ex.strike.iloc[0]:,.0f} — expiry {ex.market_end.iloc[0]:%Y-%m-%d %H:%M} UTC "
             f"— resolved {'YES' if ex.result_yes_won.iloc[0] else 'NO'}")
ax.legend(loc="upper left")
savefig(fig, "eda_example_market"); plt.show()

## EDA 2 — Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(panel.loc[usable, "price"], bins=50)
axes[0].set_title("Polymarket YES price"); axes[0].set_xlabel("price")
axes[1].hist(panel.loc[mid, "gap"].clip(-0.5, 0.5), bins=60)
axes[1].axvline(0, color="k", lw=1)
axes[1].set_title("gap = fair − price (0.05–0.95 band)"); axes[1].set_xlabel("gap")
axes[2].hist(panel.loc[usable, "stale_min"].clip(upper=30), bins=30)
axes[2].set_title("Minutes since last Polymarket price point"); axes[2].set_xlabel("minutes (clipped at 30)")
plt.tight_layout(); savefig(fig, "eda_distributions"); plt.show()

## H1 — Validity: do Polymarket prices carry the information in BTC?

*H0: Polymarket prices are unrelated to the BTC-implied probability.*
Beyond correlation, we compare **forecast quality** against the realised outcome (Brier score, lower = better)
for the market price and for the simple BTC model, overall and by time to expiry. The Brier difference has
a 95% CI from a bootstrap that resamples whole expiry hours.

In [ ]:
d = panel[usable]
r_p, p_p = stats.pearsonr(d.price, d.fair)
r_s = stats.spearmanr(d.price, d.fair).statistic
brier_poly = analysis.brier(d.result_yes_won, d.price)
brier_fair = analysis.brier(d.result_yes_won, d.fair)
bs = analysis.cluster_bootstrap_diff(analysis.brier, d.price.values, d.fair.values,
                                     d.result_yes_won.values, d.market_end.values, n_boot=config.N_BOOT)
print(f"Pearson r = {r_p:.3f} (p = {p_p:.2g}), Spearman rho = {r_s:.3f}")
print(f"Brier — Polymarket {brier_poly:.4f} | BTC model {brier_fair:.4f}")
print(f"Difference (Polymarket − model) = {bs['diff']:+.4f}, 95% CI [{bs['lo']:+.4f}, {bs['hi']:+.4f}]")

bucket = pd.cut(d.tau_min, [0, 10, 20, 40, 60, np.inf], labels=["≤10", "10–20", "20–40", "40–60", ">60"])
by_tau = d.groupby(bucket, observed=True).apply(lambda g: pd.Series({
    "n": len(g), "brier_polymarket": analysis.brier(g.result_yes_won, g.price),
    "brier_btc_model": analysis.brier(g.result_yes_won, g.fair)}))
print("\nBy minutes to settlement:"); print(by_tau.round(4))
save_metrics("h1_validity", pearson_r=r_p, spearman_rho=r_s, brier_polymarket=brier_poly,
             brier_btc_model=brier_fair, brier_diff=bs["diff"], brier_diff_lo=bs["lo"], brier_diff_hi=bs["hi"])

In [ ]:
# Reliability diagram: predicted probability vs realised YES frequency
bins = np.linspace(0, 1, 11)
fig, ax = plt.subplots(figsize=(5.5, 5))
for col, label in [("price", "Polymarket"), ("fair", "BTC model")]:
    b = pd.cut(d[col], bins, include_lowest=True)
    cal = d.groupby(b, observed=True).agg(pred=(col, "mean"), freq=("result_yes_won", "mean"))
    ax.plot(cal.pred, cal.freq, "o-", label=label)
ax.plot([0, 1], [0, 1], "k:", lw=1)
ax.set_xlabel("Predicted probability"); ax.set_ylabel("Realised YES frequency")
ax.set_title("Calibration"); ax.legend()
savefig(fig, "h1_calibration"); plt.show()

## H2 — Lead-lag between BTC information and Polymarket moves

*H0: BTC-implied probability changes are not followed by Polymarket price changes.*
`corr(d_fair_t, dp_{t+k})` pooled within markets. Mass at **k = 0** means same-minute pricing,
mass at **k > 0** means Polymarket follows BTC with a delay, mass at **k < 0** would mean Polymarket leads.
(The v1 notebook passed the series to `statsmodels.ccf` in the order that flips this sign.)

In [ ]:
ll = analysis.lead_lag(panel, "d_fair", "dp", max_lag=15, mask=mid)
pos = ll[ll.lag >= 0]
peak = pos.loc[pos["corr"].idxmax()]
share_delayed = pos.loc[pos.lag >= 1, "corr"].clip(lower=0).sum() / pos["corr"].clip(lower=0).sum()

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(ll.lag, ll["corr"], color=np.where(ll.lag >= 0, "#378ADD", "#BBBBBB"))
ax.fill_between(ll.lag, -ll.ci95, ll.ci95, color="red", alpha=.12, label="±1.96/√n (iid band)")
ax.axhline(0, color="k", lw=.8)
ax.set_xlabel("k (minutes)  —  k > 0: BTC information at t, Polymarket move at t + k")
ax.set_ylabel("corr(d_fair_t, dp_t+k)"); ax.set_title("H2: lead-lag of BTC-implied vs Polymarket changes")
ax.legend(); savefig(fig, "h2_lead_lag"); plt.show()

print(ll.set_index("lag").loc[-3:10, ["corr", "n"]].round(4).T)
print(f"\nPeak at k = {int(peak.lag)} min (corr {peak['corr']:.3f}); "
      f"share of positive correlation at k ≥ 1: {share_delayed:.1%}")
save_metrics("h2_lead_lag", corr_k0=float(ll.set_index("lag").loc[0, "corr"]), peak_lag_min=int(peak.lag),
             peak_corr=float(peak["corr"]), share_delayed=float(share_delayed),
             corr_by_lag={str(int(k)): float(v) for k, v in ll.set_index("lag")["corr"].items()})

## H3 — Does BTC volatility change how much Polymarket moves?

*H0: mean |dp| is the same in high- and low-volatility minutes.*
Median split of trailing 60-min BTC volatility, restricted to the 0.05–0.95 band (so the
0/1 ceiling cannot drive the result) and to rows where volatility is defined — v1 put
rows with **missing** volatility into the "low" group, which created its significant result.
The test is a regression with expiry-hour-clustered SE (Welch/Mann-Whitney shown for reference).

In [ ]:
h = panel[mid & panel.dp.notna() & panel.sigma_1m.notna()].copy()
h["abs_dp"] = h.dp.abs()
h["high_vol"] = (h.sigma_1m >= h.sigma_1m.median()).astype(int)
reg = analysis.mean_diff_cluster(h.abs_dp, h.high_vol, h.market_end)
ref = analysis.two_group_test(h.loc[h.high_vol == 1, "abs_dp"], h.loc[h.high_vol == 0, "abs_dp"])
print(f"mean |dp|: high vol {ref['mean_a']:.4f} (n={ref['n_a']:,}) | low vol {ref['mean_b']:.4f} (n={ref['n_b']:,})")
print(f"Clustered regression: diff = {reg['diff']:+.4f}, t = {reg['t']:.2f}, p = {reg['p_value']:.3g}")
print(f"Reference (iid, overstated): Welch p = {ref['welch_p']:.3g}, Mann-Whitney p = {ref['mwu_p']:.3g}")

fig, ax = plt.subplots(figsize=(5, 4))
g = h.groupby("high_vol").abs_dp
ax.bar(["Low vol", "High vol"], g.mean(), yerr=1.96 * g.sem(), color=["#378ADD", "#D85A30"], capsize=6)
ax.set_ylabel("Mean |dp| per minute"); ax.set_title(f"H3 (clustered p = {reg['p_value']:.3g})")
savefig(fig, "h3_volatility"); plt.show()
save_metrics("h3_volatility", mean_abs_dp_high=ref["mean_a"], mean_abs_dp_low=ref["mean_b"],
             diff=reg["diff"], t_cluster=reg["t"], p_cluster=reg["p_value"], n=reg["n"])

## Reading the results

* **H1** tells whether Polymarket prices are informative and how they compare with a mechanical BTC model.
  A Polymarket Brier score *worse* than the model means public BTC information is not fully used.
* **H2** gives a first, model-free view of timing; notebook 03 measures the delay properly.
* **H3** is descriptive; the volatility effect on *speed* is estimated in notebook 03.

All numbers are written to `results/metrics.json`; the README quotes them from there.

In [ ]:
# Optional: commit outputs back to GitHub (needs Colab secrets GITHUB_TOKEN, GIT_NAME, GIT_EMAIL)
PUSH = False
if PUSH:
    from src.colab import push
    push("results: notebook 02 (EDA + H1-H3)", paths=('figures', 'results'))